# RANBP17_exp — Spec-Driven UMAP runs

Write *what data to show* and *how to color it* as plain Python dicts, then run.
The helper `submit_runs()` (in `manuscript/ranbp17_exp_run_helpers.py`)
generates matching `FigureConfig` + `PlotConfig` classes, writes them to two
`_generated.py` files alongside the helper, and prints the bsub commands.

**Defaults** come from `RANBP17_exp_BaseFigureConfig` /
`RANBP17_exp_BasePlotConfig` (in `manuscript/`):
- `EXPERIMENT_TYPE='RANBP17_exp'`, `CELL_LINES=['iW11']`
- 3 markers (`DAPI`, `TDP-43`, `RANBP17`), up to 4 rep colors, 6 condition colors

**Spec keys you can override per run:**

| `data` key | `plot` key |
|---|---|
| `name` (required) | `color_by` (required: `rep`/`batch`/`condition`/`cell_line`/`cell_line_condition`/`marker`) |
| `input_folders` | `umap_type` (default chosen from `color_by`) |
| `cell_lines`, `conditions` | `size`, `alpha`, `figsize` |
| `markers`, `markers_to_exclude` | `color_overrides` ({item: '#hex'}) — patches auto colors |
| `add_rep_to_label`, `add_batch_to_label` | `color_mappings` — fully replaces COLOR_MAPPINGS |
| `show_ari`, `saveroot_infix`, `experiment_type` | |

`submit_runs(runs, memory=10000, dir='$run_type', submit=True, batch=BATCH)` —
pass `batch=BATCH` to auto-set `input_folders` and namespace class names per batch
(e.g. `"AllCond"` → `"Batch2_AllCond"`), preventing cross-batch collisions in the
generated files. Flip `submit=True` to actually launch bsubs.

In [11]:
import os, sys

NOVA_HOME = "/home/projects/hornsteinlab/giliwo/NOVA"
os.environ["NOVA_HOME"] = NOVA_HOME
sys.path.insert(0, NOVA_HOME)
sys.path.insert(0, os.path.dirname(NOVA_HOME))  # so `NOVA.manuscript.X` also resolves
print("NOVA_HOME:", NOVA_HOME)

BATCH = "batch3" # batch1 / batch2 / batch3
# Imports — helper, condition lists, defaults
from manuscript.ranbp17_exp_run_helpers import submit_runs
from manuscript.manuscript_figures_data_config_RANBP17_exp import (
    RANBP17_exp_ALL_CONDITIONS,
    RANBP17_exp_ALL_KD_CONDITIONS,
    RANBP17_exp_ALL_CONTROL_CONDITIONS,
    RANBP17_exp_MARKERS,
    RANBP17_exp_CELL_LINES,
)

if BATCH == "batch2":
    # for batch2 we do not have untreated condition, so we remove it from the conditions lists
    RANBP17_exp_ALL_CONDITIONS = [cond for cond in RANBP17_exp_ALL_CONDITIONS if cond != "untreated"]
    RANBP17_exp_ALL_CONTROL_CONDITIONS = [cond for cond in RANBP17_exp_ALL_CONTROL_CONDITIONS if cond != "untreated"]

print("Conditions:        ", RANBP17_exp_ALL_CONDITIONS)
print("KD conditions:     ", RANBP17_exp_ALL_KD_CONDITIONS)
print("Control conditions:", RANBP17_exp_ALL_CONTROL_CONDITIONS)
print("Markers:           ", RANBP17_exp_MARKERS)
print("Cell lines:        ", RANBP17_exp_CELL_LINES)

NOVA_HOME: /home/projects/hornsteinlab/giliwo/NOVA
Conditions:         ['tardp-kd', 'ranbp17-kd', 'control-179', 'control-180', 'both-kd', 'untreated']
KD conditions:      ['tardp-kd', 'ranbp17-kd', 'both-kd']
Control conditions: ['control-179', 'control-180', 'untreated']
Markers:            ['DAPI', 'TDP-43', 'RANBP17', 'SG', 'RANBP17_SG']
Cell lines:         ['iW11']


In [12]:
# One-time setup. NOVA_HOME points to the giliwo NOVA clone for the helpers' imports;
# the runnable + model live under the Collaboration NOVA tree.
import os, sys

NOVA_HOME = "/home/projects/hornsteinlab/Collaboration/NOVA"
MODEL_PATH = "/home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen"
OUTDIR_NAME = "RANBP17_exp_UMAPS_logs/" + BATCH
os.environ["NOVA_HOME"] = NOVA_HOME
os.environ["MODEL_PATH"] = MODEL_PATH
print(os.getcwd())
sys.path.insert(0, os.getcwd())
sys.path.insert(1, NOVA_HOME)

print("NOVA_HOME:", NOVA_HOME)
print("Python:   ", sys.executable)

/home/projects/hornsteinlab/giliwo
NOVA_HOME: /home/projects/hornsteinlab/Collaboration/NOVA
Python:    /home/projects/hornsteinlab/giliwo/.conda/envs/nova/bin/python


## Example 1 — UMAP0 by condition

All 6 conditions in a single UMAP, colored by condition (hand-picked palette
from `RANBP17_exp_BasePlotConfig`).

In [13]:
runs_all_cond = [
    {"data": {"name": "AllCond", "conditions": RANBP17_exp_ALL_CONDITIONS},
     "plot": {"color_by": "condition", "umap_type": 0, "size": 5}},
     {"data": {"name": "ControlCond", "conditions": RANBP17_exp_ALL_CONTROL_CONDITIONS},
     "plot": {"color_by": "condition", "umap_type": 0, "size": 5}},
]
submit_runs(
    runs_all_cond, memory=10000,
    dir=f"./{OUTDIR_NAME}/ranbp17_umap0_all_cond",
    submit=True, nova_home=NOVA_HOME, model_path=MODEL_PATH,
    batch=BATCH,
);

Data: +0 new, 2 already present (file now has 65 classes)
Plot: +0 new, 2 already present (file now has 64 classes)
  data classes kept (not regenerated): ['RANBP17_exp_Data_Batch3_AllCond', 'RANBP17_exp_Data_Batch3_ControlCond']
  plot classes kept (not regenerated): ['RANBP17_exp_Plot_Batch3_AllCond_condition', 'RANBP17_exp_Plot_Batch3_ControlCond_condition']
"bsub -q short -R rusage[mem=10000] -o ./RANBP17_exp_UMAPS_logs/batch3/ranbp17_umap0_all_cond/RANBP17_exp_Data_Batch3_AllCond_0.out -J umap_0 python /home/projects/hornsteinlab/Collaboration/NOVA/runnables/generate_umaps_and_plot.py /home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen ./NOVA/manuscript/manuscript_figures_data_config_RANBP17_exp_generated/RANBP17_exp_Data_Batch3_AllCond ./NOVA/manuscript/manuscript_plot_config_RANBP17_exp_generated/RANBP17_exp_Plot_Batch3_AllCond_condition"
"bsub -q short -R rusage[mem=10000] -o ./RANBP17_exp_UMAPS_logs/batch3/

Memory reservation is (MB): 10000
Memory Limit is (MB): 10000



Job <164322> is submitted to queue <short>.

>>> submitting: bsub -q short -R rusage[mem=10000] -o ./RANBP17_exp_UMAPS_logs/batch3/ranbp17_umap0_all_cond/RANBP17_exp_Data_Batch3_ControlCond_1.out -J umap_1 python /home/projects/hornsteinlab/Collaboration/NOVA/runnables/generate_umaps_and_plot.py /home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen ./NOVA/manuscript/manuscript_figures_data_config_RANBP17_exp_generated/RANBP17_exp_Data_Batch3_ControlCond ./NOVA/manuscript/manuscript_plot_config_RANBP17_exp_generated/RANBP17_exp_Plot_Batch3_ControlCond_condition


## Example 2 — UMAP1 (multi-marker), all conditions, with and without DAPI

RANBP17_exp has only 3 markers, so `_woDAPI` collapses the run to TDP-43 + RANBP17.

In [14]:
runs_umap1 = [
    {"data": {"name": "AllCond_AllMarkers"},
     "plot": {"color_by": "marker", "umap_type": 1, "size": 5}},
    {"data": {"name": "AllCond_woDAPI",
              "markers_to_exclude": ["DAPI"]},
     "plot": {"color_by": "marker", "umap_type": 1, "size": 5}},
]
submit_runs(
    runs_umap1, memory=20000,
    dir=f"./{OUTDIR_NAME}/ranbp17_umap1",
    submit=True, nova_home=NOVA_HOME, model_path=MODEL_PATH,
    batch=BATCH,
);

Data: +0 new, 2 already present (file now has 65 classes)
Plot: +0 new, 2 already present (file now has 64 classes)
  data classes kept (not regenerated): ['RANBP17_exp_Data_Batch3_AllCond_AllMarkers', 'RANBP17_exp_Data_Batch3_AllCond_woDAPI']
  plot classes kept (not regenerated): ['RANBP17_exp_Plot_Batch3_AllCond_AllMarkers_marker', 'RANBP17_exp_Plot_Batch3_AllCond_woDAPI_marker']
"bsub -q short -R rusage[mem=20000] -o ./RANBP17_exp_UMAPS_logs/batch3/ranbp17_umap1/RANBP17_exp_Data_Batch3_AllCond_AllMarkers_0.out -J umap_0 python /home/projects/hornsteinlab/Collaboration/NOVA/runnables/generate_umaps_and_plot.py /home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen ./NOVA/manuscript/manuscript_figures_data_config_RANBP17_exp_generated/RANBP17_exp_Data_Batch3_AllCond_AllMarkers ./NOVA/manuscript/manuscript_plot_config_RANBP17_exp_generated/RANBP17_exp_Plot_Batch3_AllCond_AllMarkers_marker"
"bsub -q short -R rusage[mem

## Example 2b — UMAP1 colored by condition × marker

TDP-43 and RANBP17 together in one UMAP, each dot colored by `condition_marker`.
DAPI excluded. Hand-picked colors from `COLOR_MAPPINGS_RANBP17_EXP_CONDITIONS_MARKERS`:
- **red** `ranbp17-kd_TDP-43` · **orange** `ranbp17-kd_RANBP17`
- **blue** `control-179_TDP-43` · **cyan** `control-179_RANBP17`

In [15]:
runs_umap1_cond_marker = [
    {"data": {"name": "CondMarker_ranbp17kd_ctrl179",
              "conditions": ["ranbp17-kd", "control-179"],
              "markers_to_exclude": ["DAPI"]},
     "plot": {"color_by": "condition_marker", "umap_type": 1, "size": 5}},
]
submit_runs(
    runs_umap1_cond_marker, memory=20000,
    dir=f"./{OUTDIR_NAME}/ranbp17_umap1_cond_marker",
    submit=True, nova_home=NOVA_HOME, model_path=MODEL_PATH,
    batch=BATCH,
);

Data: +0 new, 1 already present (file now has 65 classes)
Plot: +0 new, 1 already present (file now has 64 classes)
  data classes kept (not regenerated): ['RANBP17_exp_Data_Batch3_CondMarker_ranbp17kd_ctrl179']
  plot classes kept (not regenerated): ['RANBP17_exp_Plot_Batch3_CondMarker_ranbp17kd_ctrl179_condition_marker']
"bsub -q short -R rusage[mem=20000] -o ./RANBP17_exp_UMAPS_logs/batch3/ranbp17_umap1_cond_marker/RANBP17_exp_Data_Batch3_CondMarker_ranbp17kd_ctrl179_0.out -J umap_0 python /home/projects/hornsteinlab/Collaboration/NOVA/runnables/generate_umaps_and_plot.py /home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen ./NOVA/manuscript/manuscript_figures_data_config_RANBP17_exp_generated/RANBP17_exp_Data_Batch3_CondMarker_ranbp17kd_ctrl179 ./NOVA/manuscript/manuscript_plot_config_RANBP17_exp_generated/RANBP17_exp_Plot_Batch3_CondMarker_ranbp17kd_ctrl179_condition_marker"

>>> submitting: bsub -q short -R rus

## Example 3 — QC: UMAP0 by reps and by batches

In [16]:
runs_qc = [
    {"data": {"name": "AllData_ByReps"},    "plot": {"color_by": "rep",   "size": 5}},
    {"data": {"name": "AllData_ByBatches"}, "plot": {"color_by": "batch", "size": 5}},
]
submit_runs(
    runs_qc, memory=24000,
    dir=f"./{OUTDIR_NAME}/ranbp17_umap0_per_reps_and_batches",
    submit=True, nova_home=NOVA_HOME, model_path=MODEL_PATH,
    batch=BATCH,
);

Data: +0 new, 2 already present (file now has 65 classes)
Plot: +0 new, 2 already present (file now has 64 classes)
  data classes kept (not regenerated): ['RANBP17_exp_Data_Batch3_AllData_ByReps', 'RANBP17_exp_Data_Batch3_AllData_ByBatches']
  plot classes kept (not regenerated): ['RANBP17_exp_Plot_Batch3_AllData_ByReps_rep', 'RANBP17_exp_Plot_Batch3_AllData_ByBatches_batch']
"bsub -q short -R rusage[mem=24000] -o ./RANBP17_exp_UMAPS_logs/batch3/ranbp17_umap0_per_reps_and_batches/RANBP17_exp_Data_Batch3_AllData_ByReps_0.out -J umap_0 python /home/projects/hornsteinlab/Collaboration/NOVA/runnables/generate_umaps_and_plot.py /home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen ./NOVA/manuscript/manuscript_figures_data_config_RANBP17_exp_generated/RANBP17_exp_Data_Batch3_AllData_ByReps ./NOVA/manuscript/manuscript_plot_config_RANBP17_exp_generated/RANBP17_exp_Plot_Batch3_AllData_ByReps_rep"
"bsub -q short -R rusage[mem

## Example 4 — Binary coloring: control vs. KD

Every control well one color, every KD well another, regardless of the specific
condition. `untreated` is left out — add it back manually if you want to see
where it lands.

`color_overrides` is built from `RANBP17_exp_ALL_CONTROL_CONDITIONS` —
anything in that list gets `CTRL_COLOR`, KD wells get `KD_COLOR`.

In [17]:
CTRL_COLOR = "#1F77B4"  # blue  → controls
KD_COLOR   = "#D62728"  # red   → all KDs

ctrl_kd_conds = RANBP17_exp_ALL_KD_CONDITIONS + RANBP17_exp_ALL_CONTROL_CONDITIONS
controls_set = set(RANBP17_exp_ALL_CONTROL_CONDITIONS)
overrides = {c: (CTRL_COLOR if c in controls_set else KD_COLOR) for c in ctrl_kd_conds}

runs_binary = [
    {"data": {"name": "BinaryCtrlKD", "conditions": ctrl_kd_conds},
     "plot": {"color_by": "condition", "umap_type": 0, "size": 30,
              "color_overrides": overrides}},
]
submit_runs(
    runs_binary, memory=10000,
    dir=f"./{OUTDIR_NAME}/ranbp17_umap0_binary_ctrl_kd",
    submit=True, nova_home=NOVA_HOME, model_path=MODEL_PATH,
    batch=BATCH,
);

Data: +0 new, 1 already present (file now has 65 classes)
Plot: +0 new, 1 already present (file now has 64 classes)
  data classes kept (not regenerated): ['RANBP17_exp_Data_Batch3_BinaryCtrlKD']
  plot classes kept (not regenerated): ['RANBP17_exp_Plot_Batch3_BinaryCtrlKD_condition']
"bsub -q short -R rusage[mem=10000] -o ./RANBP17_exp_UMAPS_logs/batch3/ranbp17_umap0_binary_ctrl_kd/RANBP17_exp_Data_Batch3_BinaryCtrlKD_0.out -J umap_0 python /home/projects/hornsteinlab/Collaboration/NOVA/runnables/generate_umaps_and_plot.py /home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen ./NOVA/manuscript/manuscript_figures_data_config_RANBP17_exp_generated/RANBP17_exp_Data_Batch3_BinaryCtrlKD ./NOVA/manuscript/manuscript_plot_config_RANBP17_exp_generated/RANBP17_exp_Plot_Batch3_BinaryCtrlKD_condition"

>>> submitting: bsub -q short -R rusage[mem=10000] -o ./RANBP17_exp_UMAPS_logs/batch3/ranbp17_umap0_binary_ctrl_kd/RANBP17_exp_

## Example 5 — Each KD vs. each control (distinct UMAPs)

For every `(kd, control)` pair, build an independent UMAP that contains only
those two conditions. KD is always red and the chosen control always blue,
so plots are visually comparable across pairs. With 3 KDs × 2 controls this
produces 6 runs.

In [18]:
# Narrow the lists below if you only want a subset.
KDS_FILTER      = None  # e.g. ["tardp-kd"] for a single KD
CONTROLS_FILTER = None  # e.g. ["control-179"] for a single control

CTRL_COLOR = "#1F77B4"
KD_COLOR   = "#D62728"

kds   = list(RANBP17_exp_ALL_KD_CONDITIONS)
ctrls = list(RANBP17_exp_ALL_CONTROL_CONDITIONS)
if KDS_FILTER      is not None: kds   = [k for k in kds   if k in KDS_FILTER]
if CONTROLS_FILTER is not None: ctrls = [c for c in ctrls if c in CONTROLS_FILTER]

runs_kd_vs_ctrl = []
for kd in kds:
    for ctrl in ctrls:
        runs_kd_vs_ctrl.append({
            "data": {
                "name": f"{kd}_vs_{ctrl}",
                "conditions": [kd, ctrl],
            },
            "plot": {
                "color_by": "condition", "umap_type": 0, "size": 10,
                "color_overrides": {kd: KD_COLOR, ctrl: CTRL_COLOR},
            },
        })

print(f"Built {len(runs_kd_vs_ctrl)} (KD, control) UMAP runs.")

submit_runs(
    runs_kd_vs_ctrl, memory=10000,
    dir=f"./{OUTDIR_NAME}/ranbp17_umap0_kd_vs_ctrl",
    submit=True, nova_home=NOVA_HOME, model_path=MODEL_PATH,
    batch=BATCH,
);

Built 9 (KD, control) UMAP runs.
Data: +0 new, 9 already present (file now has 65 classes)
Plot: +0 new, 9 already present (file now has 64 classes)
  data classes kept (not regenerated): ['RANBP17_exp_Data_Batch3_tardp_kd_vs_control_179', 'RANBP17_exp_Data_Batch3_tardp_kd_vs_control_180', 'RANBP17_exp_Data_Batch3_tardp_kd_vs_untreated', 'RANBP17_exp_Data_Batch3_ranbp17_kd_vs_control_179', 'RANBP17_exp_Data_Batch3_ranbp17_kd_vs_control_180', 'RANBP17_exp_Data_Batch3_ranbp17_kd_vs_untreated', 'RANBP17_exp_Data_Batch3_both_kd_vs_control_179', 'RANBP17_exp_Data_Batch3_both_kd_vs_control_180', 'RANBP17_exp_Data_Batch3_both_kd_vs_untreated']
  plot classes kept (not regenerated): ['RANBP17_exp_Plot_Batch3_tardp_kd_vs_control_179_condition', 'RANBP17_exp_Plot_Batch3_tardp_kd_vs_control_180_condition', 'RANBP17_exp_Plot_Batch3_tardp_kd_vs_untreated_condition', 'RANBP17_exp_Plot_Batch3_ranbp17_kd_vs_control_179_condition', 'RANBP17_exp_Plot_Batch3_ranbp17_kd_vs_control_180_condition', 'RANBP17

## Example 6 — Highlight a condition

Color everything else gray, highlight the chosen condition in red. Handy for
pointing out where one KD lands relative to all the others.

In [19]:
to_highlight = "ranbp17-kd"
gray = "#CCCCCC"
overrides = {c: gray for c in RANBP17_exp_ALL_CONDITIONS}
overrides[to_highlight] = "#E41A1C"

runs_highlight = [
    {"data": {"name": f"AllCond_Highlight_{to_highlight}"},
     "plot": {"color_by": "condition", "umap_type": 0, "size": 5,
              "color_overrides": overrides}},
]
submit_runs(
    runs_highlight, memory=10000,
    dir=f"./{OUTDIR_NAME}/ranbp17_highlight",
    submit=True, nova_home=NOVA_HOME, model_path=MODEL_PATH,
    batch=BATCH,
);

Data: +0 new, 1 already present (file now has 65 classes)
Plot: +0 new, 1 already present (file now has 64 classes)
  data classes kept (not regenerated): ['RANBP17_exp_Data_Batch3_AllCond_Highlight_ranbp17_kd']
  plot classes kept (not regenerated): ['RANBP17_exp_Plot_Batch3_AllCond_Highlight_ranbp17_kd_condition']
"bsub -q short -R rusage[mem=10000] -o ./RANBP17_exp_UMAPS_logs/batch3/ranbp17_highlight/RANBP17_exp_Data_Batch3_AllCond_Highlight_ranbp17_kd_0.out -J umap_0 python /home/projects/hornsteinlab/Collaboration/NOVA/runnables/generate_umaps_and_plot.py /home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen ./NOVA/manuscript/manuscript_figures_data_config_RANBP17_exp_generated/RANBP17_exp_Data_Batch3_AllCond_Highlight_ranbp17_kd ./NOVA/manuscript/manuscript_plot_config_RANBP17_exp_generated/RANBP17_exp_Plot_Batch3_AllCond_Highlight_ranbp17_kd_condition"

>>> submitting: bsub -q short -R rusage[mem=10000] -o ./RA

## Your runs go here

Add cells below with your own spec lists, copy patterns from the examples above.
Each `submit_runs(...)` call **appends** to the two `_generated.py` files —
classes whose names already exist are kept untouched, so previously-submitted
bsub jobs stay safe. Pass `reset=True` to wipe both files.

In [20]:
# Template — edit and run.
my_runs = [
    # {"data": {"name": "MyRun_1", "conditions": ["ranbp17-kd", "control-179"]},
    #  "plot": {"color_by": "condition", "umap_type": 0}},
]
# submit_runs(my_runs, memory=10000, dir="my_runs", submit=True);